In [33]:
import pandas as pd
from math import floor
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import apriori

In [34]:
data = pd.read_csv("data.csv", encoding = 'unicode_escape')
data = data.dropna()
display(data.head())
display(data.shape[0])
data = data[:floor(data.shape[0]/10)] # Usando 1/10 das transações
display(data.shape[0])

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


406829

40682

In [35]:
transactions = list(data.groupby("InvoiceNo")["Description"].apply(list).to_dict().values())
display(transactions[0])

['WHITE HANGING HEART T-LIGHT HOLDER',
 'WHITE METAL LANTERN',
 'CREAM CUPID HEARTS COAT HANGER',
 'KNITTED UNION FLAG HOT WATER BOTTLE',
 'RED WOOLLY HOTTIE WHITE HEART.',
 'SET 7 BABUSHKA NESTING BOXES',
 'GLASS STAR FROSTED T-LIGHT HOLDER']

In [36]:
te = TransactionEncoder()
oht_ary = te.fit(transactions).transform(transactions, sparse=True)
sparse_df = pd.SparseDataFrame(oht_ary, columns=te.columns_, default_fill_value=False)
sparse_df

/home/abnersuniga/.local/lib/python3.7/site-packages/ipykernel_launcher.py:3: FutureWarning: SparseDataFrame is deprecated and will be removed in a future version.
Use a regular DataFrame whose columns are SparseArrays instead.

See http://pandas.pydata.org/pandas-docs/stable/user_guide/sparse.html#migrating for more.

  This is separate from the ipykernel package so we can avoid doing imports until
/home/abnersuniga/.local/lib/python3.7/site-packages/pandas/core/sparse/frame.py:257: FutureWarning: SparseSeries is deprecated and will be removed in a future version.
Use a Series with sparse values instead.

    >>> series = pd.Series(pd.SparseArray(...))

See http://pandas.pydata.org/pandas-docs/stable/user_guide/sparse.html#migrating for more.

  sparse_index=BlockIndex(N, blocs, blens),
/home/abnersuniga/.local/lib/python3.7/site-packages/pandas/core/sparse/frame.py:745: FutureWarning: SparseDataFrame is deprecated and will be removed in a future version.
Use a regular DataFrame whose

,4 PURPLE FLOCK DINNER CANDLES,OVAL WALL MIRROR DIAMANTE,SET 2 TEA TOWELS I LOVE LONDON,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,...,YULETIDE IMAGES GIFT WRAP SET,YULETIDE IMAGES S/6 PAPER BOXES,ZINC FINISH 15CM PLANTER POTS,ZINC HEART LATTICE 2 WALL PLANTER,ZINC HEART LATTICE CHARGER LARGE,ZINC HEART LATTICE CHARGER SMALL,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2465,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2466,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2467,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2468,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [37]:
%timeit fpgrowth(sparse_df, min_support=0.01, use_colnames=True)

1.13 s ± 254 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [38]:
%timeit apriori(sparse_df, min_support=0.01, use_colnames=True, low_memory=True)

41.6 s ± 2.48 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [48]:
rules_fpgrowth = fpgrowth(sparse_df, min_support=0.01, use_colnames=True) 

In [49]:
rules_fpgrowth.sort_values('support', ascending=False)

,support,itemsets
0,0.127935,(WHITE HANGING HEART T-LIGHT HOLDER)
278,0.091903,(REGENCY CAKESTAND 3 TIER)
85,0.068826,(HEART OF WICKER SMALL)
230,0.064372,(HAND WARMER BABUSHKA DESIGN)
140,0.063563,(SCOTTIE DOG HOT WATER BOTTLE)
...,...,...
557,0.010121,"(NATURAL SLATE HEART CHALKBOARD , HEART OF WIC..."
96,0.010121,(ORGANISER WOOD ANTIQUE WHITE )
692,0.010121,"(PLASTERS IN TIN STRONGMAN, PLASTERS IN TIN SP..."
629,0.010121,"(PHOTO FRAME CORNICE, WHITE HANGING HEART T-LI..."


In [50]:
rules_apriori = apriori(sparse_df, min_support=0.01, use_colnames=True, low_memory=True)

In [51]:
rules_apriori.sort_values('support', ascending=False)

,support,itemsets
456,0.127935,(WHITE HANGING HEART T-LIGHT HOLDER)
347,0.091903,(REGENCY CAKESTAND 3 TIER)
188,0.068826,(HEART OF WICKER SMALL)
172,0.064372,(HAND WARMER BABUSHKA DESIGN)
372,0.063563,(SCOTTIE DOG HOT WATER BOTTLE)
...,...,...
153,0.010121,(FIRST AID TIN)
496,0.010121,"(PLEASE ONE PERSON METAL SIGN, BEWARE OF THE C..."
54,0.010121,(BLUE HAPPY BIRTHDAY BUNTING)
29,0.010121,(ANTIQUE GLASS PEDESTAL BOWL)
